# Lesson 87 · Quantization and local inference — squeeze a 7B model onto your laptop: 8-bit affine quantization and lightweight inference with Hugging Face

**Learning goals**
- Understand how 8-bit affine quantization works (scale + zero_point)
- Implement uint8 quantization/dequantization from scratch — note: this demo uses unsigned **uint8** (0–255); industrial INT8 is signed (-128..127)
- Measure quantization error vs number of bits
- (Optional) connect to a real local LLM with Hugging Face

> First see why memory can shrink, then why the error stays under control; links back to the "quantization" interview question in L01.

← **Previous lesson** · [L86 · Sampling strategies from scratch](L86_sampling.ipynb)

> Last lesson covered **sampling strategies from scratch**: temperature / top-k / top-p in pure NumPy.
> This lesson explores **quantization and local inference**.

## How does a 7B model fit on your laptop?

A few years ago, running a 7B-parameter language model took an A100 (80 GB of VRAM, roughly $10,000).
Today the same model runs smoothly in the unified memory of an M2 MacBook — all it takes is **quantization**.

**Intuition**: float32 is precise but "fat" — every number takes 4 bytes. int8 is imprecise but "slim" — every number takes just 1 byte.
Compress the model weights from float32 to int8 and memory drops straight to 1/4.

**The core question**: how can one integer "stand for" a whole interval of floating-point values?

**The answer is affine quantization**:

```
scale     = (x_max - x_min) / 255   (how big a float step each level stands for)
zero_point = round(-x_min / scale)   (which integer level 0.0 maps to)
x_q       = clip(round(x/scale) + zero_point, 0, 255)  → uint8
x_deq     = (x_q - zero_point) × scale                 → float32 (approximate reconstruction)
```

**Quantization error**: each value loses at most scale/2 of precision — at 8 bits that's about range/510, very small.
Drop to 4 bits (16 levels) and the error grows about 16×, which is why INT4 quantization needs special calibration.

```
float32: 7B × 4B = 28 GB → needs an A100
int8:    7B × 1B =  7 GB → an RTX 3080 is enough
int4:    7B × 0.5B = 3.5 GB → an ordinary gaming GPU will do
```

This section walks through the full uint8 affine-quantization pipeline, and you'll practice by implementing the analysis function `batch_quantize_error`.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def quantize_int8(x: np.ndarray):
    """Quantize float32 weights to uint8 (0..255, asymmetric quantization); returns (quantized, scale, zero_point).
    Note: the return type is uint8 (unsigned, range 0-255), not signed int8 (-128..127)."""
    x_min, x_max = x.min(), x.max()
    scale = (x_max - x_min) / 255.0  # uint8 range 0-255 (unsigned)
    if scale < 1e-8:
        return np.zeros_like(x, dtype=np.uint8), 1.0, 0
    zero_point = int(np.clip(round(-x_min / scale), 0, 255))
    x_q = np.round(x / scale).astype(np.int32) + zero_point
    x_q = np.clip(x_q, 0, 255).astype(np.uint8)
    return x_q, scale, zero_point

def dequantize_int8(x_q: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    """Dequantize: uint8 → float32."""
    return (x_q.astype(np.float32) - zero_point) * scale

# Test
np.random.seed(0)
W = np.random.randn(64, 64).astype(np.float32)   # a simulated weight matrix

try:
    W_q, scale, zp = quantize_int8(W)
    assert W_q is not None, "⬜ not implemented yet"
    assert W_q.dtype == np.uint8, f"quantized result should be uint8, got {W_q.dtype}"
    assert W_q.shape == W.shape, f"quantized result should have the input's shape, got {W_q.shape}"

    W_deq = dequantize_int8(W_q, scale, zp)
    error = np.abs(W - W_deq)

    print(f'original range: [{W.min():.3f}, {W.max():.3f}]')
    print(f'Scale: {scale:.6f}, Zero point: {zp}')
    print(f'quantization error: mean={error.mean():.4f}, max={error.max():.4f}')
    print(f'memory: float32={W.nbytes} bytes → uint8={W_q.nbytes} bytes  ({W.nbytes/W_q.nbytes:.1f}× compression)')

    assert error.mean() < 0.05, f"quantization error too large: {error.mean():.4f}"
    print(f'✅ quantization verified, mean error={error.mean():.4f}')
except (NotImplementedError, TypeError):
    print('⬜ not implemented yet')

> **⚠ A note on data types**: this lesson's demo uses `np.uint8` (unsigned integers, range 0–255) for affine quantization.
> In industry, INT8 usually means **signed** `int8` (range -128–127) with symmetric quantization (zero_point=0).
> Both have the same precision (8 bits); the difference is:
> - `uint8`: suits value ranges skewed positive (e.g. activations after ReLU, ≥ 0)
> - `int8`: suits value ranges symmetric about zero (e.g. linear-layer weights)
>
> Switching this lesson's code to `np.int8` + symmetric quantization is left as an after-class exercise.

In [ ]:
# Error vs number of bits
print(f'{"bits":>4}  {"levels":>7}  {"mean err":>10}  {"max err":>10}')
print('-' * 40)
for bits in [8, 6, 4, 3, 2]:
    levels = 2**bits - 1
    scale_b = (W.max() - W.min()) / levels
    W_q_b = np.clip(np.round((W - W.min()) / scale_b), 0, levels)
    W_r = W_q_b * scale_b + W.min()  # equivalent form: subtract x_min directly, no explicit zero_point
    err = np.abs(W - W_r)
    print(f'{bits:>4}  {levels:>7}  {err.mean():>10.4f}  {err.max():>10.4f}')

## ✏️ Exercise: `batch_quantize_error(W, bits_list)`

An everyday job for a quantization engineer: compare the quantization error of one weight matrix at different bit widths, and decide which bit width strikes the best balance between precision and memory.

**Signature**:
```python
def batch_quantize_error(W: np.ndarray, bits_list: list) -> dict:
    # returns {bits: {"scale": float, "mean_err": float, "max_err": float}}
```

**Three steps to implement it**:

| Step | Action | Code hint |
|---|---|---|
| 1 | Compute scale for each bits value | `scale = (W.max()-W.min()) / (2**bits - 1)` |
| 2 | Quantize + dequantize | `W_q = clip(round((W-W.min())/scale), 0, 2**bits-1)` → `W_r = W_q*scale + W.min()` |
| 3 | Compute and return the error statistics | `{"scale": scale, "mean_err": abs_err.mean(), "max_err": abs_err.max()}` |

**Acceptance criteria**:
- `mean_err` for `bits=8` is < 0.05 (8 bits means small error)
- `mean_err` for `bits=2` > `mean_err` for `bits=8` (fewer bits, larger error)
- the return value is a dict keyed by the integer bits

In [ ]:
def batch_quantize_error(W: np.ndarray, bits_list: list) -> dict:
    """Compare quantization error across bit widths.

    Args:
        W: float32 weight matrix
        bits_list: list of bit widths, e.g. [8, 6, 4, 3, 2]

    Returns:
        {bits: {"scale": float, "mean_err": float, "max_err": float}}
    """
    # TODO: loop over bits_list and compute the affine-quantization error for each bits value
    raise NotImplementedError("Implement batch_quantize_error")


# ── Check ─────────────────────────────────────────────────────────────────────
np.random.seed(0)
W_test = np.random.randn(64, 64).astype(np.float32)

try:
    result = batch_quantize_error(W_test, [8, 4, 2])
    assert isinstance(result, dict), f"should return a dict, got {type(result)}"
    assert 8 in result and 2 in result, f"result is missing keys, got {list(result.keys())}"
    assert result[2]["mean_err"] > result[8]["mean_err"], \
        f"fewer bits should mean larger error: bits=2 mean_err={result[2]['mean_err']:.4f} should be > bits=8 {result[8]['mean_err']:.4f}"
    assert result[8]["mean_err"] < 0.05, f"8-bit error should be < 0.05, got {result[8]['mean_err']:.4f}"
    print("✅ batch_quantize_error verified:")
    for bits, stats in sorted(result.items()):
        print(f"  bits={bits}: scale={stats['scale']:.4f}, mean_err={stats['mean_err']:.4f}, max_err={stats['max_err']:.4f}")
except (NotImplementedError, TypeError):
    print("⚠️ batch_quantize_error not implemented yet")

## ✏️ Closed-book derivation check cell — the maths of quantization

**Rule: collapse every cell above and do the derivation below from memory alone.**

**Problem**: a weight matrix has range `[-2.0, 2.0]` and is quantized with uint8 (0–255) affine quantization.

1. Compute `scale` (to 4 decimal places)
2. Compute `zero_point` (an integer, so that 0.0 maps exactly)
3. What is `x_q`, the result of quantizing `x = 0.5`?
4. What float value do you get by dequantizing `x_q = 200`?
5. What is the theoretical upper bound on the quantization error? (in terms of scale)

(Write your derivation here...)

In [ ]:
# ✏️ Self-review for this lesson
l87_review = {
    "quantization_intuition":    None,  # understand quantization = integer levels standing for a float interval, scale=range/levels? True/False
    "affine_formula":            None,  # can write the full scale/zero_point/quantize/dequantize formulas? True/False
    "batch_error_impl":          None,  # batch_quantize_error() correct and its check passes? True/False
    "bit_tradeoff":              None,  # understand fewer bits → bigger scale → bigger error, and that 4× compression comes from float32→int8? True/False
    "whiteboard_passed":         None,  # whiteboard challenge (quantization maths) done? True/False
}

unfilled = [k for k, v in l87_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l87_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L87 fully cleared! On to L88: TF-IDF retrieval from scratch')

In [ ]:
# Check: the whiteboard problem by hand
import numpy as np

x_min, x_max = -2.0, 2.0
bits = 8
levels = 2**bits - 1  # 255

scale_ans = (x_max - x_min) / levels
zp_ans = int(round(-x_min / scale_ans))
x_q_ans = int(np.clip(round(0.5 / scale_ans) + zp_ans, 0, 255))
x_deq_ans = (200 - zp_ans) * scale_ans
max_error_ans = scale_ans / 2.0

print(f"1. scale = ({x_max}-({x_min}))/{levels} = {scale_ans:.4f}")
print(f"2. zero_point = round(-({x_min})/{scale_ans:.4f}) = {zp_ans}")
print(f"3. x_q for x=0.5: round(0.5/{scale_ans:.4f})+{zp_ans} = {x_q_ans}")
print(f"4. x_deq for x_q=200: (200-{zp_ans})×{scale_ans:.4f} = {x_deq_ans:.4f}")
print(f"5. quantization error bound = scale/2 = {max_error_ans:.4f}")

assert abs(scale_ans - 4.0/255) < 1e-9
assert zp_ans == 128
assert abs(x_deq_ans - (200 - 128) * 4/255) < 1e-6
print("✅ quantization-maths derivation verified")

## Optional: local inference with Hugging Face

> The code below needs `transformers` installed and suitable model weights.
> If you skip it, the quantization section above has already shown the core algorithm in full.

⚠️ **Note**: `model.generate()` is a Hugging Face wrapper, used here purely as engineering integration; the core inference algorithms (KV cache, sampling) are in L85–L86.

In [ ]:
# Engineering-integration example (needs transformers)
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch

    # Demo with a small model (swap in a local model you have)
    MODEL_ID = 'HuggingFaceTB/SmolLM2-360M'   # ~0.7 GB, runs locally

    print(f'Loading {MODEL_ID}...')
    # Uncomment to actually run it:
    # tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    # model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    # inputs = tokenizer('The Fourier transform', return_tensors='pt')
    # with torch.no_grad():
    #     out = model.generate(**inputs, max_new_tokens=30, temperature=0.8, do_sample=True)
    # print(tokenizer.decode(out[0], skip_special_tokens=True))
    print('(commented out — uncomment it once you have the model weights)')
    print('The core algorithms (KV cache, sampling) are implemented from scratch in L85 and L86')

except ImportError:
    print('transformers not installed — skipping the engineering-integration part')
    print('The quantization part above (8-bit quantize/dequantize) has already run in full')

## Wrap-up

| Concept | What to remember |
|---|---|
| 8-bit quantization (uint8 affine demo) | scale = range/255, error ≈ scale/2 |
| 4× memory compression | float32 → int8: video memory (VRAM) needs drop to 1/4 |
| Trade-off | fewer bits → smaller model → larger quantization error |
| Engineering integration | Hugging Face = a convenient wrapper, not a substitute for understanding it from scratch |
| L88 | RAG retrieval — smart question answering without needing a big model |


---

→ **Next lesson** · [L88 · TF-IDF retrieval from scratch](L88_tfidf_retrieval.ipynb)

> Next lesson covers **TF-IDF retrieval from scratch**: term frequency × inverse document frequency, vector search in NumPy.